> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 9 – Modelos preditivos e validação honesta

**Objetivo:** prever argila a partir do espectro e comparar modelos **sem vazamento de dados**.

**Regra de ouro:** o modelo é escolhido pela validação cruzada no treino. O teste é usado uma única vez, no final.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import preparar_nir
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor

meta, X, wl = preparar_nir(n=1500)
y = pd.to_numeric(meta['Clay_gkg'], errors='coerce').to_numpy(float)
ok = np.isfinite(y)
X, y = X[ok], y[ok]
print(X.shape)

## Exercício 1 – Métricas
Implemente sem usar bibliotecas prontas:
- `rmse`: raiz da média dos erros ao quadrado.
- `rpiq`: (Q3 − Q1 do observado) ÷ RMSE.
- `ccc`: 2·cov ÷ (var_obs + var_prev + (média_obs − média_prev)²).

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

def rpiq(y, p):
    q1, q3 = np.percentile(y, [25, 75])
    return float((q3 - q1) / rmse(y, p))

def ccc(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    cov = np.mean((y - y.mean()) * (p - p.mean()))
    return float(2 * cov / (y.var() + p.var() + (y.mean() - p.mean()) ** 2))

print(rmse([1, 2, 3], [1, 2, 4]), ccc([1, 2, 3], [1, 2, 3]))

## Exercício 2 – Escolher o modelo pela validação cruzada

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
modelos = {
    'PLSR': Pipeline([('sc', StandardScaler()), ('m', PLSRegression(8))]),
    'RF': RandomForestRegressor(200, random_state=42, n_jobs=-1),
}
kf = KFold(5, shuffle=True, random_state=42)
cv = {n: cross_val_score(m, X_tr, y_tr, cv=kf, scoring='r2').mean() for n, m in modelos.items()}
melhor = max(cv, key=cv.get)
print(cv, '-> melhor:', melhor)

## Exercício 3 – Avaliar no teste (uma única vez)

In [ ]:
from sklearn.metrics import r2_score
modelo = modelos[melhor].fit(X_tr, y_tr)
pred = np.ravel(modelo.predict(X_te))
print(f'R²={r2_score(y_te, pred):.3f} RMSE={rmse(y_te, pred):.1f} '
      f'RPIQ={rpiq(y_te, pred):.2f} CCC={ccc(y_te, pred):.3f} viés={np.mean(pred - y_te):.1f}')

lim = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
plt.scatter(y_te, pred, s=10, alpha=0.6); plt.plot(lim, lim, 'k--')
plt.xlabel('Observado (g/kg)'); plt.ylabel('Previsto (g/kg)'); plt.gca().set_aspect('equal')
plt.title(f'{melhor} – argila'); plt.show()

## Desafio – Um modelo por cluster
Use o K-means da aula 8 (sobre os PCs do treino) e treine um PLSR por cluster. Compare o RMSE no teste com o modelo global.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
pca = PCA(6).fit(X_tr)
km = KMeans(3, n_init=20, random_state=42).fit(pca.transform(X_tr))
c_tr, c_te = km.labels_, km.predict(pca.transform(X_te))
pred_local = np.empty_like(y_te)
for c in np.unique(c_tr):
    m = Pipeline([('sc', StandardScaler()), ('m', PLSRegression(8))]).fit(X_tr[c_tr == c], y_tr[c_tr == c])
    if np.any(c_te == c):
        pred_local[c_te == c] = np.ravel(m.predict(X_te[c_te == c]))
print('RMSE global:', round(rmse(y_te, pred), 1), '| RMSE por cluster:', round(rmse(y_te, pred_local), 1))

## Checkpoint

Por que escolher o modelo pelo R² do teste deixa o resultado otimista? Por que o `StandardScaler` precisa estar dentro do `Pipeline`?